# v3.15 — the production notebook, run end to end with regions set

Fetches `vp/tryon_er.ipynb` from the branch and executes its own Install, Downloads, Inputs, Load and Pipeline cells unmodified, then drives its functions over every product shot at `upper` and `lower` and five worn garments at both. Prints PASS/FAIL per case and a total. Drive-free; any GPU with 20 GB works. Runtime → Run all.

## 1 · Settings and budget

In [ ]:
import glob
import hashlib
import json
import os
import re
import shutil
import time
import traceback
import urllib.request
import zipfile

REPO = "101011101/magichour_takehome"
BRANCH = "v3.3-lock"
TEST_SEED = 46
RUN = "/content/v315"
CASES = json.loads('[{"id": "g001__upper", "kind": "product", "garment": "g001", "garment_path": "test_set1/garments/g001.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g001__lower", "kind": "product", "garment": "g001", "garment_path": "test_set1/garments/g001.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g002__upper", "kind": "product", "garment": "g002", "garment_path": "test_set1/garments/g002.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g002__lower", "kind": "product", "garment": "g002", "garment_path": "test_set1/garments/g002.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g003__upper", "kind": "product", "garment": "g003", "garment_path": "test_set1/garments/g003.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g003__lower", "kind": "product", "garment": "g003", "garment_path": "test_set1/garments/g003.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g006__upper", "kind": "product", "garment": "g006", "garment_path": "test_set1/garments/g006.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g006__lower", "kind": "product", "garment": "g006", "garment_path": "test_set1/garments/g006.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g007__upper", "kind": "product", "garment": "g007", "garment_path": "test_set1/garments/g007.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g007__lower", "kind": "product", "garment": "g007", "garment_path": "test_set1/garments/g007.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g008__upper", "kind": "product", "garment": "g008", "garment_path": "test_set1/garments/g008.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g008__lower", "kind": "product", "garment": "g008", "garment_path": "test_set1/garments/g008.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g010__upper", "kind": "product", "garment": "g010", "garment_path": "test_set1/garments/g010.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "ghost_mannequin - no person, must skip the bald pass and force full"}, {"id": "g010__lower", "kind": "product", "garment": "g010", "garment_path": "test_set1/garments/g010.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "ghost_mannequin - no person, must skip the bald pass and force full"}, {"id": "g016__upper", "kind": "product", "garment": "g016", "garment_path": "test_set1/garments/g016.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g016__lower", "kind": "product", "garment": "g016", "garment_path": "test_set1/garments/g016.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g017__upper", "kind": "product", "garment": "g017", "garment_path": "test_set1/garments/g017.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "ghost_mannequin - no person, must skip the bald pass and force full"}, {"id": "g017__lower", "kind": "product", "garment": "g017", "garment_path": "test_set1/garments/g017.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "ghost_mannequin - no person, must skip the bald pass and force full"}, {"id": "g019__upper", "kind": "product", "garment": "g019", "garment_path": "test_set1/garments/g019.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g019__lower", "kind": "product", "garment": "g019", "garment_path": "test_set1/garments/g019.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g020__upper", "kind": "product", "garment": "g020", "garment_path": "test_set1/garments/g020.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g020__lower", "kind": "product", "garment": "g020", "garment_path": "test_set1/garments/g020.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g021__upper", "kind": "product", "garment": "g021", "garment_path": "test_set1/garments/g021.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g021__lower", "kind": "product", "garment": "g021", "garment_path": "test_set1/garments/g021.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g022__upper", "kind": "product", "garment": "g022", "garment_path": "test_set1/garments/g022.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g022__lower", "kind": "product", "garment": "g022", "garment_path": "test_set1/garments/g022.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g023__upper", "kind": "product", "garment": "g023", "garment_path": "test_set1/garments/g023.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g023__lower", "kind": "product", "garment": "g023", "garment_path": "test_set1/garments/g023.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g025__upper", "kind": "product", "garment": "g025", "garment_path": "test_set1/garments/g025.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g025__lower", "kind": "product", "garment": "g025", "garment_path": "test_set1/garments/g025.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g026__upper", "kind": "product", "garment": "g026", "garment_path": "test_set1/garments/g026.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g026__lower", "kind": "product", "garment": "g026", "garment_path": "test_set1/garments/g026.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "flat_lay - no person, must skip the bald pass and force full"}, {"id": "g028__upper", "kind": "product", "garment": "g028", "garment_path": "test_set1/garments/g028.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "upper", "why": "ghost_mannequin - no person, must skip the bald pass and force full"}, {"id": "g028__lower", "kind": "product", "garment": "g028", "garment_path": "test_set1/garments/g028.jpg", "person": "p001", "person_path": "test_set1/people/p001.jpg", "region": "lower", "why": "ghost_mannequin - no person, must skip the bald pass and force full"}, {"id": "g004__upper", "kind": "worn", "garment": "g004", "garment_path": "test_set1/garments/g004.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "upper", "why": "tight top, an ordinary band cut"}, {"id": "g004__lower", "kind": "worn", "garment": "g004", "garment_path": "test_set1/garments/g004.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "lower", "why": "tight top, an ordinary band cut"}, {"id": "g024__upper", "kind": "worn", "garment": "g024", "garment_path": "test_set1/garments/g024.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "upper", "why": "lower-body garment, where lower is the meaningful half"}, {"id": "g024__lower", "kind": "worn", "garment": "g024", "garment_path": "test_set1/garments/g024.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "lower", "why": "lower-body garment, where lower is the meaningful half"}, {"id": "g015__upper", "kind": "worn", "garment": "g015", "garment_path": "test_set1/garments/g015.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "upper", "why": "a dress - ATR labels it one class over the whole body"}, {"id": "g015__lower", "kind": "worn", "garment": "g015", "garment_path": "test_set1/garments/g015.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "lower", "why": "a dress - ATR labels it one class over the whole body"}, {"id": "g030__upper", "kind": "worn", "garment": "g030", "garment_path": "test_set1/garments/g030.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "upper", "why": "no in-frame hip on the archived bald frame - expected to fall back"}, {"id": "g030__lower", "kind": "worn", "garment": "g030", "garment_path": "test_set1/garments/g030.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "lower", "why": "no in-frame hip on the archived bald frame - expected to fall back"}, {"id": "p019__upper", "kind": "worn", "garment": "p019", "garment_path": "test_set1/people/p019.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "upper", "why": "waist-up wearer - the lower band is likely too small, a second fallback path"}, {"id": "p019__lower", "kind": "worn", "garment": "p019", "garment_path": "test_set1/people/p019.jpg", "person": "p006", "person_path": "test_set1/people/p006.jpg", "region": "lower", "why": "waist-up wearer - the lower band is likely too small, a second fallback path"}]')

n_product = sum(1 for c in CASES if c["kind"] == "product")
n_worn = sum(1 for c in CASES if c["kind"] == "worn")
bald_calls = n_worn
tryon_calls = len(CASES)
estimate = bald_calls * 1.48 + tryon_calls * 1.9 + (n_worn + n_product // 2) * 0.58
print(f"{len(CASES)} cases: {n_product} product-shot requests, {n_worn} worn-garment requests")
print(f"klein calls expected: {bald_calls} bald passes + {tryon_calls} try-ons = {bald_calls + tryon_calls}")
print(f"generation estimate on an A100: ~{estimate / 60:.1f} min, plus the weight download and load")

## 2 · Fetch the shipped notebook and find its sections

In [ ]:
api = f"https://api.github.com/repos/{REPO}/commits?path=vp/tryon_er.ipynb&sha={BRANCH}&per_page=1"
try:
    with urllib.request.urlopen(api, timeout=30) as r:
        COMMIT = json.load(r)[0]["sha"]
except Exception as e:
    COMMIT = None
    print(f"could not resolve the commit ({e}); testing the branch head instead")
RAW = f"https://raw.githubusercontent.com/{REPO}/{COMMIT or BRANCH}"
os.makedirs(f"{RUN}/shipped", exist_ok=True)
urllib.request.urlretrieve(f"{RAW}/vp/tryon_er.ipynb", f"{RUN}/shipped/tryon_er.ipynb")
SHIPPED_SHA256 = hashlib.sha256(open(f"{RUN}/shipped/tryon_er.ipynb", "rb").read()).hexdigest()
shipped = json.load(open(f"{RUN}/shipped/tryon_er.ipynb"))

SECTIONS = ["## 1 · Install", "## 2 · Downloads", "## 3 · Inputs", "## 4 · Load", "## 5 · Pipeline"]
SKIPPED = ["## 6a · Run", "## 6b · Run", "## 7 · Output"]


def section_cells(nb):
    found = {}
    cells = nb["cells"]
    for i, c in enumerate(cells):
        if c["cell_type"] != "markdown":
            continue
        title = "".join(c["source"]).strip()
        for s in SECTIONS + SKIPPED:
            if title.startswith(s):
                if i + 1 >= len(cells) or cells[i + 1]["cell_type"] != "code":
                    raise RuntimeError(f"section {s!r} is not followed by a code cell")
                found[s] = "".join(cells[i + 1]["source"])
    missing = [s for s in SECTIONS + SKIPPED if s not in found]
    if missing:
        raise RuntimeError(f"tryon_er.ipynb no longer has the sections this test drives: {missing}")
    return found


CELLS = section_cells(shipped)
for s in SECTIONS:
    name = re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
    open(f"{RUN}/shipped/{name}.py", "w").write(CELLS[s])
print(f"tryon_er.ipynb at {COMMIT or BRANCH}  sha256 {SHIPPED_SHA256[:16]}")
print("executing, unmodified:", ", ".join(SECTIONS))
print("skipped (they wait for an upload):", ", ".join(SKIPPED))

## 3 · Run the shipped Install cell

In [ ]:
exec(compile(CELLS["## 1 · Install"], "tryon_er.ipynb · 1 Install", "exec"), globals())

## 4 · Run the shipped Downloads, Inputs, Load and Pipeline cells

In [ ]:
for s in ("## 2 · Downloads", "## 3 · Inputs", "## 4 · Load", "## 5 · Pipeline"):
    t = time.perf_counter()
    exec(compile(CELLS[s], f"tryon_er.ipynb · {s[3:]}", "exec"), globals())
    print(f"{s[3:]:14s} {time.perf_counter() - t:7.1f} s")
GPU = torch.cuda.get_device_name(0)
if "A100" not in GPU:
    print(f"{GPU}: not an A100 - this tests behaviour, so it still counts; timings are not comparable")
print("GPU", GPU, "| MAX_RES default", MAX_RES)

## 5 · The test photos, from the repo

In [ ]:
os.makedirs(f"{RUN}/inputs", exist_ok=True)
for path in sorted({c["garment_path"] for c in CASES} | {c["person_path"] for c in CASES}):
    dst = f"{RUN}/inputs/" + os.path.basename(path)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(f"{RAW}/{path}", dst)
    if cv2.imread(dst) is None:
        raise RuntimeError(f"unreadable input {path}")
print(len(os.listdir(f"{RUN}/inputs")), "photos from the repo at", COMMIT or BRANCH)

## 6 · The harness: a pass-through recorder on klein, and the expectations

In [ ]:
_klein_shipped = klein
CALLS = []


def klein(images, prompt, seed, size):
    t = time.perf_counter()
    out = _klein_shipped(images, prompt, seed, size)
    CALLS.append({"kind": "bald" if prompt == BALD_PROMPT else "try-on", "prompt": prompt,
                  "seed": int(seed), "size": [int(size[0]), int(size[1])],
                  "seconds": round(time.perf_counter() - t, 3)})
    return out


def head_pixels(why):
    m = re.search(r"(\d+)", why or "")
    return int(m.group(1)) if m else None


def judge(case, info, prep_calls, tryon, cached):
    fails, notes = [], []
    req, kind = case["region"], case["kind"]
    route, applied, fallback = info.get("route"), info.get("region"), info.get("fallback", "")
    bald = any(c["kind"] == "bald" for c in prep_calls)
    sent = tryon[0]["prompt"] if tryon else None
    wanted = ER_PROMPT if applied == "full" else ER_REGION.get(applied)
    if len(tryon) != 1:
        fails.append(f"{len(tryon)} generations in try_on, expected 1")
    if sent != wanted:
        fails.append(f"call 2 was sent the wrong sentence for region {applied!r}")
    if kind == "product":
        if route != "product":
            fails.append(f"the gate saw a person on a product shot ({info.get('route_why')})")
        if bald:
            fails.append("the bald pass ran on a product shot")
        if applied != "full":
            fails.append(f"a product shot was cut to {applied!r} instead of forced to full")
        if info.get("requested") != req:
            notes.append(f"the garment record reads requested={info.get('requested')!r}, not {req!r}:"
                         " the forcing is visible through route=product, not through requested")
    else:
        if route != "worn":
            fails.append(f"the gate missed the wearer ({info.get('route_why')})")
        if not bald and not cached:
            fails.append("a worn garment was prepared without the bald pass")
        if applied not in ("full", req):
            fails.append(f"applied region {applied!r} is neither {req!r} nor full")
        if applied == "full" and not fallback:
            fails.append(f"{req} was silently treated as full, with no recorded reason")
        if applied == req and "kept_fraction" not in info:
            fails.append("the band was applied but kept_fraction was not recorded")
        if fallback:
            notes.append(f"fell back to full: {fallback}")
    if cached:
        notes.append("the reference came from the cache (same garment, same applied region, same route)")
    return fails, notes


def run_case(case):
    rec = {"case": case, "pass": False, "fails": [], "notes": []}
    t0 = time.perf_counter()
    try:
        garment = load_image(f"{RUN}/inputs/" + os.path.basename(case["garment_path"]), "garment")
        person = load_image(f"{RUN}/inputs/" + os.path.basename(case["person_path"]), "person")
        CALLS.clear()
        reference, info, garment_times, cached = prepare_garment(garment, case["region"])
        prep_calls = list(CALLS)
        CALLS.clear()
        result, tryon_times = try_on(person, reference, TEST_SEED, info["region"], MAX_RES)
        tryon = list(CALLS)
        fails, notes = judge(case, info, prep_calls, tryon, cached)
        cv2.imwrite(f"{RUN}/refs/{case['id']}.jpg", reference, [cv2.IMWRITE_JPEG_QUALITY, 92])
        cv2.imwrite(f"{RUN}/gen/{case['id']}.jpg", result, [cv2.IMWRITE_JPEG_QUALITY, 92])
        rec.update({
            "pass": not fails, "fails": fails, "notes": notes,
            "actual": {"gate": info.get("route") == "worn", "head_px": head_pixels(info.get("route_why")),
                       "route_why": info.get("route_why"), "route": info.get("route"),
                       "bald_pass": any(c["kind"] == "bald" for c in prep_calls),
                       "requested": case["region"], "applied": info.get("region"),
                       "fallback": info.get("fallback", ""), "kept_fraction": info.get("kept_fraction"),
                       "head_route": info.get("head_route"), "cached": bool(cached),
                       "call2": "region" if tryon and tryon[0]["prompt"] in ER_REGION.values() else "full",
                       "output": f"{result.shape[1]}x{result.shape[0]}",
                       "reference": f"{reference.shape[1]}x{reference.shape[0]}"},
            "times": {**{"garment " + k: round(v, 3) for k, v in garment_times.items()},
                      **{"person " + k: round(v, 3) for k, v in tryon_times.items()}},
            "klein_calls": prep_calls + tryon,
        })
    except Exception as e:
        rec["fails"] = [f"raised {type(e).__name__}: {e}"]
        rec["traceback"] = traceback.format_exc()
    rec["expected"] = ({"gate": False, "route": "product", "bald_pass": False, "applied": "full", "call2": "full"}
                       if case["kind"] == "product" else
                       {"gate": True, "route": "worn", "bald_pass": True,
                        "applied": f"{case['region']} (or full with a recorded fallback)",
                        "call2": "region (or full with a recorded fallback)"})
    rec["wall_seconds"] = round(time.perf_counter() - t0, 2)
    return rec

## 7 · Run every case

In [ ]:
shutil.rmtree(CACHE_DIR, ignore_errors=True)
for d in ("refs", "gen"):
    os.makedirs(f"{RUN}/{d}", exist_ok=True)
RECORDS = []
for i, case in enumerate(CASES, 1):
    rec = run_case(case)
    RECORDS.append(rec)
    a = rec.get("actual", {})
    print(f"{i:2d}/{len(CASES)} {'PASS' if rec['pass'] else 'FAIL'}  {case['id']:14s} "
          f"route={a.get('route')} bald={a.get('bald_pass')} applied={a.get('applied')} call2={a.get('call2')}"
          + (f"  <- {rec['fails'][0]}" if rec["fails"] else ""))
    json.dump(RECORDS, open(f"{RUN}/records.json", "w"), indent=1)

## 8 · PASS / FAIL

In [ ]:
width = max(len(r["case"]["id"]) for r in RECORDS)
print(f"{'case':{width}s}  kind     req    route    bald   applied  call2   head_px  result")
for r in sorted(RECORDS, key=lambda r: (r["pass"], r["case"]["kind"], r["case"]["id"])):
    a = r.get("actual", {})
    print(f"{r['case']['id']:{width}s}  {r['case']['kind']:7s}  {r['case']['region']:5s}  "
          f"{str(a.get('route')):7s}  {str(a.get('bald_pass')):5s}  {str(a.get('applied')):7s}  "
          f"{str(a.get('call2')):6s}  {str(a.get('head_px')):7s}  {'PASS' if r['pass'] else 'FAIL'}")
    for f in r["fails"]:
        print(f"{'':{width}s}    FAIL: {f}")
passed = sum(r["pass"] for r in RECORDS)
for kind in ("product", "worn"):
    group = [r for r in RECORDS if r["case"]["kind"] == kind]
    print(f"{kind:8s} {sum(r['pass'] for r in group)}/{len(group)} pass")
klein_total = sum(len(r.get("klein_calls", [])) for r in RECORDS)
print(f"TOTAL    {passed}/{len(RECORDS)} pass · {klein_total} klein calls · tryon_er.ipynb at {COMMIT or BRANCH}")
json.dump({"commit": COMMIT, "branch": BRANCH, "shipped_sha256": SHIPPED_SHA256, "gpu": GPU,
           "max_res": MAX_RES, "seed": TEST_SEED, "passed": passed, "cases": len(RECORDS),
           "klein_calls": klein_total}, open(f"{RUN}/meta.json", "w"), indent=1)

## 9 · Zip, download, release the GPU

In [ ]:
TERMINATE_WHEN_DONE = True  #@param {type:"boolean"}
DOWNLOAD_GRACE_SECONDS = 120  #@param {type:"integer"}

name = f"v315_prod_smoke_{time.strftime('%Y%m%d_%H%M')}"
zip_path = f"/content/{name}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("inputs", "refs", "gen", "shipped"):
        for f in sorted(os.listdir(f"{RUN}/{sub}")):
            z.write(f"{RUN}/{sub}/{f}", f"{sub}/{f}")
    for f in ("records.json", "meta.json"):
        z.write(f"{RUN}/{f}", f)
with zipfile.ZipFile(zip_path) as z:
    if z.testzip() is not None:
        raise RuntimeError("the zip is corrupt")
    names = z.namelist()
print(f"{name}.zip · {len(names)} files · {os.path.getsize(zip_path) / 1e6:.1f} MB")

from google.colab import files
downloaded = False
try:
    files.download(zip_path)
    downloaded = True
except Exception as e:
    print(f"download failed ({e}); the zip is at {zip_path} and the runtime is kept")
if downloaded:
    print(f"downloading; waiting {DOWNLOAD_GRACE_SECONDS}s before the runtime is released")
    time.sleep(DOWNLOAD_GRACE_SECONDS)
    if TERMINATE_WHEN_DONE:
        from google.colab import runtime
        runtime.unassign()